#### **Connect to git repo and install requirements**

In [ ]:
import os

#check if the repository folder already exists on the Colab disk
if os.path.exists('/content/QML4EO-reproduction'):
    print("Repo found! Pulling latest changes from GitHub..")
    %cd /content/QML4EO-reproduction
    !git pull
else:
    print("Cloning repo for the first time..")
    %cd /content
    !git clone https://github.com/yeshapan/QML4EO-reproduction.git
    %cd QML4EO-reproduction

#install required dependencies
!pip install -r requirements.txt -q

Cloning repo for the first time..
/content
Cloning into 'QML4EO-reproduction'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 59 (delta 18), reused 51 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 6.80 MiB | 15.16 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/QML4EO-reproduction
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 85.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
from src.baselines.cnn import set_seed, train_baseline
from src.utils.data_loader import get_eurosat_dataloaders
from src.models.hqcnn import HybridQCNN

#lock seed for perfect reproducibility across classical and quantum operations
set_seed(42)

#setup device to use the T4 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware utilized: {device}")

Hardware utilized: cuda


#### **Load Data**

In [3]:
#load data (keeping 64x64 to match classical baseline comparison)
train_loader, val_loader, classes = get_eurosat_dataloaders(
    data_dir="./data",
    batch_size=32,
    img_size=64
)

Downloading/Loading EuroSAT dataset into ./data...


100%|██████████| 94.3M/94.3M [00:01<00:00, 81.0MB/s]


Dataset loaded successfully!
Total images: 27000 | Training: 21600 | Validation: 5400
Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']


#### **Initialize + Train the Hybrid Quantum Model**

In [4]:
#initialize the hybrid quantum model
#4 qubits, 1 quantum layer (standard phi-lab baseline)
model = HybridQCNN(num_classes=len(classes), num_qubits=4, num_layers=1)
model = model.to(device)

#calculate total trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters (Hybrid QCNN): {total_params:,}")

#execute the training loop!
#we reuse the classical training loop because pennylane integrates perfectly with pytorch
print("\nStarting Hybrid Quantum Training:")
history = train_baseline(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=0.001,
    device=device
)

print("\nHybrid training complete!")

Total Trainable Parameters (Hybrid QCNN): 5,274

Starting Hybrid Quantum Training:


Epoch 1/5 [Train]: 100%|██████████| 675/675 [00:27<00:00, 24.29it/s, loss=1.4813]


Epoch 1 Summary -> Train Loss: 1.8669 | Val Accuracy: 39.50%


Epoch 2/5 [Train]: 100%|██████████| 675/675 [00:25<00:00, 26.68it/s, loss=1.5696]


Epoch 2 Summary -> Train Loss: 1.4874 | Val Accuracy: 41.63%


...

Epoch 3 Summary -> Train Loss: 1.4026 | Val Accuracy: 43.54%


Epoch 4/5 [Train]: 100%|██████████| 675/675 [00:25<00:00, 26.27it/s, loss=1.3487]


Epoch 4 Summary -> Train Loss: 1.3585 | Val Accuracy: 47.50%


Epoch 5/5 [Train]: 100%|██████████| 675/675 [00:25<00:00, 26.56it/s, loss=1.3264]


Epoch 5 Summary -> Train Loss: 1.3240 | Val Accuracy: 47.56%

Hybrid training complete!


**The Parameter Discrepancy: Why is this Quantum Model (with 4 qubits) smaller than baseline CNN model?**

It is counter-intuitive that adding a quantum circuit reduces the total parameter count (5,418 classical vs 5,274 hybrid). This occurs due to the **information bottleneck** required to fit classical data into a small quantum state.

* **Classical CNN Final Layers (330 parameters):**
    * The fully connected layer maps 32 channels directly to 10 classes: (32 x 10 weights) + 10 biases = 330 parameters.

* **Hybrid QCNN Final Layers (186 parameters):**
    * **Bottleneck Layer:** Maps 32 channels down to 4 qubits: (32 x 4 weights) + 4 biases = 132 parameters.
    * **Quantum Ansatz:** 1 layer across 4 qubits requires 4 parameters (one trainable rotation per qubit).
    * **Final Classification Layer:** Maps 4 qubits to 10 classes: (4 x 10 weights) + 10 biases = 50 parameters

* **Math:** 330 - 186 = 144 
    Subtracting 144 from the classical total of 5,418 yields exactly 5,274 trainable parameters. The lower accuracy (47.56%) is a direct result of forcing the model to compress its learned features through this tiny 4-qubit bottleneck in only 5 epochs.